# Automatic Differentiation with PyPoli

This tutorial demonstrates how to use JAX's automatic differentiation capabilities with PyPoli to optimize quantum circuits. This is equivalent to the `8-automatic-differentiation.ipynb` example in PauliPropagation.jl.

We will implement a Variational Quantum Eigensolver (VQE) to find the ground state energy of a Transverse Field Ising Model (TFIM).

In [ ]:
import jax
import jax.numpy as jnp
import sys
import os

# Add the source directory to the path to import pypoli
sys.path.append(os.path.abspath("../src"))

from pypoli import Circuit, PauliString, expectation_value
from pypoli import RX, RY, RZ, RZZ, CNOT, CZ, H

# Enable 64-bit precision for better numerical stability
jax.config.update("jax_enable_x64", True)

## 1. Define the Hamiltonian

We will consider a 1D Transverse Field Ising Model (TFIM) with open boundary conditions.
The Hamiltonian is given by:
$$ H = -J \sum_{i=0}^{N-2} Z_i Z_{i+1} - h \sum_{i=0}^{N-1} X_i $$

We will choose $N=4$, $J=1.0$, and $h=1.0$.

In [ ]:
N = 4
J = 1.0
h = 1.0

hamiltonian = []

# Interaction terms Z_i Z_{i+1}
for i in range(N - 1):
    term = PauliString({i: 'Z', i+1: 'Z'}, -J)
    hamiltonian.append(term)

# Transverse field terms X_i
for i in range(N):
    term = PauliString({i: 'X'}, -h)
    hamiltonian.append(term)

print(f"Hamiltonian has {len(hamiltonian)} terms.")
for term in hamiltonian:
    print(term)

## 2. Define the Ansatz (Parameterized Circuit)

We will use a "Hardware Efficient Ansatz" consisting of alternating layers of single-qubit rotations and entangling gates.

- **Single-qubit gates**: $R_y(\theta)$
- **Entangling gates**: CNOTs between adjacent qubits

In [ ]:
def ansatz_circuit(params, n_qubits, depth):
    """
    Constructs a hardware efficient ansatz circuit.
    
    Args:
        params: Flat array of parameters.
        n_qubits: Number of qubits.
        depth: Number of layers.
        
    Returns:
        Circuit object.
    """
    circuit = Circuit()
    param_idx = 0
    
    # Initial rotations
    for i in range(n_qubits):
        circuit.add_gate(RY(i, params[param_idx]))
        param_idx += 1
        
    for d in range(depth):
        # Entangling layer (CNOT chain)
        for i in range(n_qubits - 1):
            circuit.add_gate(CNOT(i, i+1))
            
        # Rotation layer
        for i in range(n_qubits):
            circuit.add_gate(RY(i, params[param_idx]))
            param_idx += 1
            
    return circuit

## 3. Define the Cost Function with JAX Decorators

We can use JAX decorators `@jax.jit` and `@jax.value_and_grad` to define our loss function cleanly. This automatically handles compilation and gradient computation.

The cost function is the expectation value of the Hamiltonian:
$$ E(\theta) = \langle 0 | U^\dagger(\theta) H U(\theta) | 0 \rangle $$

In [ ]:
depth = 3

@jax.jit
@jax.value_and_grad
def loss_fn(params):
    # 1. Build circuit (JAX traces this)
    circuit = ansatz_circuit(params, N, depth)
    
    total_energy = 0.0
    
    # 2. Sum expectation values for all Hamiltonian terms
    for term in hamiltonian:
        # Calculate <0| U_dag term U |0>
        val = expectation_value(circuit, term)
        total_energy += val
        
    return total_energy

## 4. Initialize and Run Optimization

We initialize parameters and run a simple gradient descent loop.

In [ ]:
n_params = N + depth * N  # Initial layer + depth * rotation layers
print(f"Number of parameters: {n_params}")

# Initialize random parameters
key = jax.random.PRNGKey(42)
params = jax.random.uniform(key, shape=(n_params,), minval=0, maxval=2*jnp.pi)

# Compute initial energy and gradient (triggers JIT compilation)
print("Compiling...")
e_init, grad_init = loss_fn(params)
print("Compiled!")
print(f"Initial Energy: {e_init:.6f}")
print(f"Gradient Norm: {jnp.linalg.norm(grad_init):.6f}")

## 5. Optimization Loop (VQE)

In [ ]:
learning_rate = 0.1
n_steps = 100
history = []

current_params = params

print("Starting optimization...")
for step in range(n_steps):
    # Direct call to JIT-compiled function
    energy, grads = loss_fn(current_params)
    history.append(energy)
    
    # Update parameters
    current_params = current_params - learning_rate * grads
    
    if step % 10 == 0:
        print(f"Step {step:3d}: Energy = {energy:.6f}")

print(f"Final Energy: {energy:.6f}")

## 6. Plotting Results

Let's visualize the convergence.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(history)
plt.xlabel("Optimization Step")
plt.ylabel("Energy")
plt.title("VQE Convergence for TFIM")
plt.grid(True)
plt.show()

## 7. Performance Measurement

Since we used `@jax.jit`, the execution should be very fast after the first call.

In [ ]:
import time

start = time.time()
for _ in range(100):
    _ = loss_fn(params)
end = time.time()

print(f"Time for 100 evaluations (JIT): {end - start:.4f} s")
print(f"Average time per step: {(end - start)/100 * 1000:.2f} ms")